# 프롬프트 실험실 (Prompt Lab)

운영 파이프라인과 완전히 분리된 노트북입니다. GitHub에 아무것도 push하지 않습니다.

사용 방법:
1. 런타임 → 런타임 유형 변경 → T4 GPU 선택
2. 셀 1, 2를 한 번만 실행 (모델 로드, 1~2분)
3. 셀 3의 `PROMPT`, `NEGATIVE_PROMPT`, `STEPS`, `GUIDANCE` 값을 자유롭게 바꿔가며
   반복 실행 → 결과를 즉시 확인 (모델은 이미 로드되어 있어 매번 1초 내)
4. 마음에 드는 조합을 찾으면 그 `PROMPT` 문구를 그대로 복사해서 전달해주세요.
   (Claude가 prompt_templates.json에 반영합니다)

In [ ]:
# 1. 환경 설정
!pip install diffusers transformers accelerate -q

import torch
from diffusers import AutoPipelineForText2Image

print("환경 설정 완료")

In [ ]:
# 2. 모델 로드 (1~2분, 한 번만 실행)
pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16"
).to("cuda")
print("모델 로드 완료")

In [ ]:
# 3. 실험 셀 - 아래 값들을 자유롭게 바꿔가며 반복 실행하세요
#    셀 1,2는 한 번만 실행하면 됩니다. 이 셀만 반복 실행하면 됩니다.
import requests, time

# GitHub에서 config 읽기 (NEGATIVE_PROMPT 기본값용)
try:
    cfg = requests.get("https://raw.githubusercontent.com/stockpipeline/stockpipeline/main/config.json").json()
    DEFAULT_NEGATIVE = cfg.get("image", {}).get("negative_prompt", "")
except:
    DEFAULT_NEGATIVE = "multiple bowls, many small dishes, grid of plates, banchan spread, tiled composition, repeated pattern"

# ── 여기를 수정하세요 ────────────────────────────────────
PROMPT = (
    "A single steaming bowl of Korean yukgaejang spicy beef soup, "
    "one bowl only, no other dishes, top-down view, "
    "professional food photography, white background"
)

NEGATIVE_PROMPT = DEFAULT_NEGATIVE  # config에서 자동으로 가져옴 (또는 직접 수정)

STEPS = 4          # 1~4 (SDXL-Turbo 권장, 높을수록 디테일↑ 속도↓)
GUIDANCE = 1.5     # 0=negative_prompt 무효, 1~3 권장
NUM_IMAGES = 4     # 한 번에 생성할 장수
# ────────────────────────────────────────────────────────

print(f"PROMPT: {PROMPT[:80]}...")
print(f"NEGATIVE: {NEGATIVE_PROMPT[:60]}...")
print(f"steps={STEPS}, guidance={GUIDANCE}, num={NUM_IMAGES}")
print()

start = time.time()
for i in range(NUM_IMAGES):
    seed = torch.randint(0, 2**31 - 1, (1,)).item()
    generator = torch.Generator(device="cuda").manual_seed(seed)
    image = pipe(
        prompt=PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE,
        width=1024,
        height=1024,
        generator=generator,
    ).images[0]
    print(f"  [{i+1}/{NUM_IMAGES}] seed={seed}, {time.time()-start:.1f}s")
    display(image)

print(f"\n완료! 총 {time.time()-start:.1f}초")
print("마음에 드는 이미지가 있으면 그 PROMPT 문구를 복사해서 Claude에게 전달해주세요.")

---
## img2img 실험 (선택 실행)

공개 이미지 URL을 레퍼런스로 넣어서 비슷한 느낌의 새 이미지를 생성합니다.
셀 1, 2(모델 로드)가 먼저 실행되어 있어야 합니다.

- `strength` 낮을수록(0.3~0.4) 원본과 비슷, 높을수록(0.6~0.8) 창의적
- CC0 공개 이미지나 직접 찍은 사진 URL을 사용하세요


In [ ]:
# img2img 실험 셀 - URL만 바꿔가며 반복 실행하세요
from diffusers import AutoPipelineForImage2Image
from PIL import Image
import requests, time
from io import BytesIO

# img2img 파이프라인 (text2img와 모델 공유, 추가 로드 없음)
pipe_i2i = AutoPipelineForImage2Image.from_pipe(pipe)

# ── 여기를 바꾸세요 ──────────────────────────────────────
REF_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Yukgaejang.jpg/1200px-Yukgaejang.jpg"

PROMPT = (
    "a single bowl of Korean yukgaejang spicy beef soup, "
    "shredded beef and vegetables in red broth, "
    "top-down view, professional food photography, "
    "white background, one bowl only"
)
# DEFAULT_NEGATIVE는 셀 3에서 정의됩니다.
# 셀 3 미실행 시 config에서 직접 로드
try:
    NEGATIVE_PROMPT = DEFAULT_NEGATIVE
except NameError:
    import requests as _r2
    _cfg2 = _r2.get("https://raw.githubusercontent.com/stockpipeline/stockpipeline/main/config.json").json()
    NEGATIVE_PROMPT = _cfg2.get("image", {}).get("negative_prompt",
        "multiple bowls, many small dishes, tiled pattern, collage")

STRENGTH   = 0.5   # 0.3(원본에 가까움) ~ 0.8(많이 변형)
STEPS      = 4     # SDXL-Turbo 권장
GUIDANCE   = 1.5
NUM_IMAGES = 4
# ────────────────────────────────────────────────────────

# 레퍼런스 이미지 로드
res = requests.get(REF_URL, timeout=10)
ref_img = Image.open(BytesIO(res.content)).convert("RGB").resize((1024, 1024))
print(f"레퍼런스 이미지 로드 완료: {ref_img.size}")
display(ref_img)
print()

# img2img 생성
start = time.time()
generated_images = []  # 생성된 이미지를 저장해두어 Drive 저장 시 재사용

for i in range(NUM_IMAGES):
    seed = torch.randint(0, 2**31 - 1, (1,)).item()
    generator = torch.Generator(device="cuda").manual_seed(seed)
    image = pipe_i2i(
        prompt=PROMPT,
        image=ref_img,
        negative_prompt=NEGATIVE_PROMPT,
        num_inference_steps=STEPS,
        strength=STRENGTH,
        guidance_scale=GUIDANCE,
        generator=generator,
    ).images[0]
    generated_images.append(image)
    print(f"[{i+1}/{NUM_IMAGES}] seed={seed}, {time.time()-start:.1f}s")
    display(image)

print(f"\n완료! 총 {time.time()-start:.1f}초")
print(f"저장하려면 아래 SAVE_INDICES에 번호를 입력하세요 (0~{NUM_IMAGES-1})")

# ── 마음에 드는 이미지를 Drive에 저장 ──────────────────
# 위 결과 중 저장하고 싶은 이미지의 인덱스를 입력하세요 (0부터 시작)
# 예: SAVE_INDICES = [0, 2]  → 1번째, 3번째 이미지 저장
# 저장하지 않으려면 SAVE_INDICES = [] 로 두세요

SAVE_INDICES = []  # ← 여기에 저장할 이미지 번호 입력

if SAVE_INDICES:
    from google.colab import drive as _drive
    import os as _os
    _drive.mount('/content/drive', force_remount=False)

    _save_dir = Path("/content/drive/MyDrive/stockpipeline/lora_dataset")
    _save_dir.mkdir(parents=True, exist_ok=True)

    _saved = 0
    for _idx in SAVE_INDICES:
        if _idx >= len(generated_images):
            print(f"  ⚠️  인덱스 {_idx}는 범위 초과 (0~{len(generated_images)-1})")
            continue
        _img   = generated_images[_idx]
        _fname = _save_dir / f"lora_train_{len(list(_save_dir.glob('*.jpg')))+1:04d}.jpg"
        _img.save(str(_fname), "JPEG", quality=95)
        print(f"  [{_idx}] 저장됨: {_fname.name}")
        _saved += 1

    print(f"\n✅ {_saved}장 저장 완료 → {_save_dir}")
    print(f"   현재 데이터셋 총 {len(list(_save_dir.glob('*.jpg')))}장")
else:
    print("저장할 이미지 없음 (SAVE_INDICES가 비어있음)")